In [2]:
import os
import warnings

# 1. Hide the USER_AGENT environment variable warning
os.environ["USER_AGENT"] = "TravelAgent/1.0"

# 2. Silence the specific LangChain and Deprecation warnings
from langchain_core._api import LangChainDeprecationWarning

warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)
#warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import asyncio
import operator
import json
import random
from typing import Annotated, Sequence, TypedDict

from langchain_core._api import LangChainDeprecationWarning
from langgraph.errors import LangGraphDeprecatedSinceV10

from langchain_chroma import Chroma
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import StateGraph, END
from langgraph.prebuilt import tools_condition, ToolNode
from langgraph.prebuilt import create_react_agent
from langgraph.managed.is_last_step import RemainingSteps


In [3]:
EMBEDDING_MODEL = 'nomic-embed-text:latest'
LOCAL_LLM = 'gemma4:e4b'
TEMPERATURE = 0.7

In [4]:
embedding = OllamaEmbeddings(model=EMBEDDING_MODEL)
llm_model = ChatOllama(
    model=LOCAL_LLM,
    temperature=TEMPERATURE,
    use_responses_api=True
)

In [5]:
async def build_vectorstore(destinations: Sequence[str]) -> Chroma:
    urls = [f'https://en.wikivoyage.org/wiki/{destination}' for destination in destinations]
    loader = AsyncHtmlLoader(urls, default_parser="html.parser")
    print("Downloading destination pages ...")
    docs = await loader.aload()

    splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=128)
    chunks = sum([splitter.split_documents([d]) for d in docs], [])

    print(f"Embedding {len(chunks)} chunks ...")
    BATCH_SIZE = 100  # Safe batch size for Ollama requests
    
    vectordb_client = Chroma.from_documents(
        documents=chunks[:BATCH_SIZE],
        embedding=embedding,
    )
    
    for i in range(BATCH_SIZE, len(chunks), BATCH_SIZE):
        batch = chunks[i:i + BATCH_SIZE]
        print(f"Processing batch {i} to {min(i + BATCH_SIZE, len(chunks))}...")
        vectordb_client.add_documents(batch)
        
    print("Vector store ready.\n")
    return vectordb_client

In [6]:
UK_DESTINATIONS = [
    'Cornwall',
    'North_Cornwall',
    'South_Cornwall',
    'West_Cornwall',
    'Truro_(England)',
    'Newquay',
    'Port_Isaac',
    'St_Ives',
]

async def get_travel_info_vectorstore() -> Chroma:
    vectorstore_client = await build_vectorstore(UK_DESTINATIONS)
    return vectorstore_client

In [7]:
ti_vectorstore_client = await get_travel_info_vectorstore()
ti_retriever = ti_vectorstore_client.as_retriever()

Fetching pages: 100%|##################################################| 8/8 [00:00<00:00, 10.51it/s]


Embedding 2326 chunks ...
Processing batch 100 to 200...
Processing batch 200 to 300...
Processing batch 300 to 400...
Processing batch 400 to 500...
Processing batch 500 to 600...
Processing batch 600 to 700...
Processing batch 700 to 800...
Processing batch 800 to 900...
Processing batch 900 to 1000...
Processing batch 1000 to 1100...
Processing batch 1100 to 1200...
Processing batch 1200 to 1300...
Processing batch 1300 to 1400...
Processing batch 1400 to 1500...
Processing batch 1500 to 1600...
Processing batch 1600 to 1700...
Processing batch 1700 to 1800...
Processing batch 1800 to 1900...
Processing batch 1900 to 2000...
Processing batch 2000 to 2100...
Processing batch 2100 to 2200...
Processing batch 2200 to 2300...
Processing batch 2300 to 2326...
Vector store ready.



In [8]:
class WeatherForecast(TypedDict):
    town: str
    weather: Literal['sunny', 'foggy', 'rainy', 'windy']
    temperature: int
    
@tool(description='Get the weather forecast given the town name.')
def weather_forecast(town: str) -> dict:
    '''Get a weather forecast for a given town.
    Returns a WeatherForecast object with weather and temperature.
    '''
    _weather_options = ['sunny', 'foggy', 'rainy', 'windy']
    _temp_min = 18
    _temp_max = 31
    
    weather = random.choice(_weather_options)
    temperature = random.randint(_temp_min, _temp_max)
    return WeatherForecast(town=town, weather=weather, temperature=temperature)
    
@tool(description='Search travel information about destinations in England.')
def search_travel_info(query: str) -> str:
    """Search embedded WikiVoyage content for 
    information about destinations in England."""    
    docs = ti_retriever.invoke(query)
    top = docs[:4] if isinstance(docs, list) else docs
    return "\n---\n".join(d.page_content for d in top)

tools = [weather_forecast, search_travel_info]    

In [9]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    remaining_steps: RemainingSteps    

In [11]:
system_prompt = '''
You are a helpful travel assistant that searches information and retrieves weather forecasts.    

CRITICAL RULES:
    1. Only suggest destinations that you have found inside the 'search_travel_info' tool.
    2. Identify candidate towns from your travel info search and check the weather for MULTIPLE candidate towns in parallel (simultaneously) to find the ones with the best weather.
    3. If your initial batch of towns has bad weather, query the weather for any backup towns in a single batch before formulating your final answer.
'''
travel_info_agent = create_react_agent(
    model=llm_model,
    tools=tools,
    state_schema=AgentState,
    prompt=system_prompt,
)

In [12]:
def chat_loop():
    print("UK Travel Assistant (type 'exit' to quit)")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"exit", "quit"}:
            break
        state = {"messages": [HumanMessage(content=user_input)]}
        
        # --- FIX HERE: Add a recursion limit config ---
        # 10 steps is more than enough for a search + weather fallback batch
        config = {"recursion_limit": 10} 
        
        try:
            result = travel_info_agent.invoke(state, config=config)
            print("\n\n\n")
            print(result)
            print("\n\n\n")
            response_msg = result["messages"][-1].content
            print(f"Assistant: {response_msg}\n")
        except GraphRecursionError:
            # Catch the limit gracefully if it hits a runaway loop
            print("\nAssistant: I'm sorry, I couldn't find any towns with ideal weather after checking several options.\n")

            

In [ ]:
chat_loop()

UK Travel Assistant (type 'exit' to quit)


You:  Suggest two Cornwall beach towns with nice weather.






{'messages': [HumanMessage(content='Suggest two Cornwall beach towns with nice weather.', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-09T19:24:46.222265Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5122402625, 'load_duration': 2116724958, 'prompt_eval_count': 228, 'prompt_eval_duration': 154846000, 'eval_count': 201, 'eval_duration': 2849083000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019eadd8-190a-7cd3-9493-e3d62c8924ba-0', tool_calls=[{'name': 'search_travel_info', 'args': {'query': 'Cornwall beach towns'}, 'id': 'a1184498-2518-47f5-86a0-5f85455aa435', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 228, 'output_tokens': 201, 'total_tokens': 429}), ToolMessage(content='<span id="Padstow" class="fn org listing-name"><a rel="mw:WikiLink" href="//en.wikivoyage.org/wiki/Padstow" title="P